In [1]:
import sys as __harness_sys
from pathlib import Path as __harness_Path
__harness_sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\tutorials')
__harness_sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\scripts')
from _offline import enforce_offline as __guard, blocked_attempts as __blocked
__guard()
import capture_runresults_213 as __capture
__capture.install()
import agentic_systems as __harness_toolkit
assert __harness_toolkit.__version__ == "2.1.3"
assert "site-packages" in __harness_Path(__harness_toolkit.__file__).parts

# Contratos, salida estructurada y juez explícito

Separa contrato de actor, esquema de salida y evaluación de calidad. Todos los ejemplos ejecutados aquí son deterministas.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [2]:
import agentic_systems as toolkit


contract declara evidencia de Tools y salida; policy declara comportamiento, runtime los límites del actor. output_schema es JSON Schema portable y sus referencias son locales. all_of, any_of, allowed, forbidden y min_count pertenecen al mismo tool_expectation; no hay helpers expect ni must_call alternativo.

In [3]:
import agentic_systems as toolkit

def contracts_example():

    @toolkit.tool
    def echo(value: int) -> dict:
        """Return a structured integer object."""
        return {'value': value}
    capability = toolkit.skill(name='echo', tools=[echo])
    schema = {'type': 'object', 'properties': {'value': {'type': 'integer'}}, 'required': ['value']}
    contract = {'tool_expectation': {'all_of': ['echo'], 'allowed': ['echo'], 'min_count': 1}, 'failure_policy': 'no_unresolved', 'output_schema': schema}
    rows = []
    for name in ('native', 'langgraph', 'openai-agents', 'strands'):
        worker = toolkit.Agent('structured_operator', runtime=toolkit.runtime(provider='python-runtime', max_turns=8, max_tool_calls=2), framework=toolkit.framework(name), skills=(capability,), contract=contract, policy={'trace': 'compact'})
        positive = worker.run({'tool': 'echo', 'input': {'value': 7}})
        negative = worker.run({'tool': 'echo', 'input': {'value': 'not-an-integer'}})
        assert positive.ok and positive.data['value'] == 7 and (not negative.ok)
        rows.append({'framework': name, 'positive': positive.to_dict(), 'negative': negative.to_dict()})
    return {'input': {'value': 7}, 'process': 'Required Tool + actor budget + portable JSON Schema validation', 'output': rows, 'validation': 'Python Runtime; no language-model structured-output claim'}
report = contracts_example()
assert len(report['output'])==4
for row in report['output']:
    print(row['framework'],row['positive']['ok'],row['negative']['ok'])
print(report['validation'])

native True False
langgraph True False
openai-agents True False
strands True False
Python Runtime; no language-model structured-output claim


Eval combina las expectativas, los checks y el juez requerido mediante AND. El juez es un Agent normal; recibe input, expected, resultado y evidencia. La rúbrica fija criterios y threshold. Los criterios fallidos requieren hallazgos con evidencia; un ok inconsistente no se acepta. Este juez es una fixture de protocolo: no valida semántica de ningún modelo.

In [4]:
import agentic_systems as toolkit

def build_episode(backend='native'):
    worker = toolkit.Agent('sum', toolkit.Action('sum', lambda d: {'total': d['previous'] + d['amount']}), backend=backend)
    return toolkit.Environment('ledger', worker, ({'amount': 2}, {'amount': 3}, {'amount': 100}), memory={'total': 0}, observe=lambda row, memory: {'amount': row['amount'], 'previous': memory['total']}, update=lambda memory, row, result: result.data, stop=lambda memory, result: memory['total'] >= 5)

from dataclasses import replace
import agentic_systems as toolkit

def evaluation_example():
    target = toolkit.Agent('calculator', toolkit.Action('increment', lambda d: {'value': d['value'] + 1}))
    judge = toolkit.Agent('fixture_judge', toolkit.Action('score', lambda payload: {'criteria': {'correctness': 1.0}, 'findings': [], 'rationale': 'Fixture: deterministic oracle confirmed the supplied case.'}))
    evaluation = toolkit.Eval('contract', target, ({'input': {'value': 3}, 'expected': {'data': {'value': 4}}},), checks=(lambda result, case: {'ok': result.data['value'] > 0, 'issues': []},), judge=judge, rubric={'criteria': ['correctness'], 'threshold': 0.9})
    positive = evaluation.run()
    negative = replace(evaluation, cases=({'input': {'value': 3}, 'expected': {'data': {'value': 999}}},)).run()
    episode = toolkit.Eval('episode_reward', build_episode(), ({'input': {}, 'expected': {'ok': True}},), checks=(lambda result, case: {'ok': result.data['summary']['memory']['total'] == 5, 'issues': [], 'metrics': {'reward': result.data['summary']['memory']['total']}},)).run()
    assert positive.ok and (not negative.ok) and episode.ok
    assert episode.data['cases'][0]['checks'][1]['metrics']['reward'] == 5
    return {'input': {'value': 3}, 'process': 'Exact expected + check + explicit fixture judge; separate episode check', 'output': positive.to_dict(), 'negative': negative.to_dict(), 'episode': episode.to_dict(), 'judge_validation': 'fixture-only; no semantic model inference'}
report = evaluation_example()
assert report['output']['ok'] and not report['negative']['ok'] and report['episode']['ok']
print({'positive':report['output']['ok'],'negative':report['negative']['ok'],'judge':report['judge_validation']})

{'positive': True, 'negative': False, 'judge': 'fixture-only; no semantic model inference'}


In [5]:
__capture.save('C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\revisions\\2.1.3\\learning-evidence-elevated\\notebook-runresults\\20261023T044738543256\\03-02_contracts_judges.json', 'tutorials/api/02_contracts_judges.ipynb')
assert __blocked() == 0